# Data Preprocessing & Feature Engineering

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

## Load Raw Data & Parse Timestamps

In [2]:
df = pd.read_csv("data/synthetic_forecast_cleaned.csv")
print(f"Raw shape: {df.shape}")
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.set_index('timestamp', inplace=True)
df.sort_index(inplace=True)
df.head()

Raw shape: (8688, 26)


,location_id,meal,year,month,day,hour,minute,day_of_week,week_of_year,is_weekend,hour_frac,hour_sin,hour_cos,month_sin,month_cos,is_holiday,holiday_name,special_event,footfall,footfall_raw,waste_kg,waste_kg_target,waste_organic_kg,waste_recyclable_kg,waste_landfill_kg
timestamp,,,,,,,,,,,,,,,,,,,,,,,,,
2025-01-01 00:00:00,b,closed,2025,1,1,0,0,2,1,0,0.0000,0.0000,1.0000,0.5000,0.8660,1,new year's day,nfl playoff watch party,0,0,0.0000,0.0000,0.0000,0.0000,0.0000
2025-01-01 00:30:00,d,closed,2025,1,1,0,30,2,1,0,0.5000,0.0000,1.0000,0.5000,0.8660,1,new year's day,nfl playoff watch party,4,4,0.0400,0.0400,0.0200,0.0100,0.0100
2025-01-01 01:00:00,d,closed,2025,1,1,1,0,2,1,0,1.0000,0.2588,0.9659,0.5000,0.8660,1,new year's day,nfl playoff watch party,1,1,0.0500,0.0500,0.0200,0.0100,0.0100
2025-01-01 01:30:00,b,closed,2025,1,1,1,30,2,1,0,1.5000,0.2588,0.9659,0.5000,0.8660,1,new year's day,nfl playoff watch party,4,4,0.0500,0.0500,0.0200,0.0100,0.0200
2025-01-01 02:00:00,d,closed,2025,1,1,2,0,2,1,0,2.0000,0.5000,0.8660,0.5000,0.8660,1,new year's day,nfl playoff watch party,0,0,0.0200,0.0200,0.0100,0.0000,0.0100


## Aggregate to Daily Level per Canteen Section

We sum waste and cost variables, take the mean foot traffic because it’s a rate per half‑hour and preserve flag columns (holiday, special event)

In [3]:
df['date'] = df.index.normalize()

agg_dict = {
    'waste_kg': 'sum',
    'waste_organic_kg': 'sum',
    'waste_recyclable_kg': 'sum',
    'waste_landfill_kg': 'sum',
    'footfall': 'mean',
    'is_holiday': 'max',
    'special_event': lambda x: ', '.join(x.dropna().unique()) if any(x.notna()) else None
}

daily = df.groupby(['date', 'location_id', 'meal']).agg(agg_dict).reset_index()

daily.rename(columns={
    'date': 'date',
    'location_id': 'section',
    'meal': 'meal',
    'waste_kg': 'waste_kg',
    'footfall': 'foot_traffic'
}, inplace=True)

print(f"Daily (by meal) shape: {daily.shape}")
display(daily.head())

Daily (by meal) shape: (2618, 10)


,date,section,meal,waste_kg,waste_organic_kg,waste_recyclable_kg,waste_landfill_kg,foot_traffic,is_holiday,special_event
0,2025-01-01,a,closed,0.1800,0.1000,0.0500,0.0400,1.6667,1,nfl playoff watch party
1,2025-01-01,a,dinner,3.1800,1.9900,0.8100,0.3900,12.5000,1,nfl playoff watch party
2,2025-01-01,a,lunch,4.2300,2.2800,1.4300,0.5200,22.6667,1,nfl playoff watch party
3,2025-01-01,b,breakfast,2.9800,1.2200,0.8900,0.8700,6.7500,1,nfl playoff watch party
4,2025-01-01,b,closed,0.2000,0.1000,0.0600,0.0300,1.8333,1,nfl playoff watch party


## Holiday Flags

In [4]:
daily['is_holiday'] = daily['is_holiday'].astype(int)
daily['has_special_event'] = daily['special_event'].notna().astype(int)
daily.drop(columns=['special_event'], inplace=True)
daily[['date', 'section', 'waste_kg', 'is_holiday', 'has_special_event']].head()

,date,section,waste_kg,is_holiday,has_special_event
0,2025-01-01,a,0.1800,1,1
1,2025-01-01,a,3.1800,1,1
2,2025-01-01,a,4.2300,1,1
3,2025-01-01,b,2.9800,1,1
4,2025-01-01,b,0.2000,1,1


## Calendar Features

features derived from date they will used directly by tree‑based models and as optional regressors for Prophet/SARIMA.

In [5]:
daily['year'] = daily['date'].dt.year
daily['month'] = daily['date'].dt.month
daily['day'] = daily['date'].dt.day
daily['day_of_week'] = daily['date'].dt.dayofweek
daily['day_of_year'] = daily['date'].dt.dayofyear
daily['week_of_year'] = daily['date'].dt.isocalendar().week
daily['quarter'] = daily['date'].dt.quarter
daily['is_weekend'] = (daily['day_of_week'] >= 5).astype(int)

print("Calendar features added.")
daily[['date', 'year', 'month', 'day_of_week', 'is_weekend']].head()

Calendar features added.


,date,year,month,day_of_week,is_weekend
0,2025-01-01,2025,1,2,0
1,2025-01-01,2025,1,2,0
2,2025-01-01,2025,1,2,0
3,2025-01-01,2025,1,2,0
4,2025-01-01,2025,1,2,0


## Cyclical Encoding

We add sin/cos transformations of hour‑of‑year and day‑of‑week for models that need it

In [6]:
daily['dow_sin'] = np.sin(2 * np.pi * daily['day_of_week'] / 7)
daily['dow_cos'] = np.cos(2 * np.pi * daily['day_of_week'] / 7)
daily['doy_sin'] = np.sin(2 * np.pi * daily['day_of_year'] / 365)
daily['doy_cos'] = np.cos(2 * np.pi * daily['day_of_year'] / 365)
daily[['date', 'day_of_week', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos']].head()

,date,day_of_week,dow_sin,dow_cos,doy_sin,doy_cos
0,2025-01-01,2,0.9749,-0.2225,0.0172,0.9999
1,2025-01-01,2,0.9749,-0.2225,0.0172,0.9999
2,2025-01-01,2,0.9749,-0.2225,0.0172,0.9999
3,2025-01-01,2,0.9749,-0.2225,0.0172,0.9999
4,2025-01-01,2,0.9749,-0.2225,0.0172,0.9999


## Base Dataset for All Models

We create `waste_features_full.csv` which contains one row per (date, section) with all features except lags and rolling stats

In [7]:
full_cols = [
    'date', 'section',
    'waste_kg', 'foot_traffic',
    'waste_organic_kg', 'waste_recyclable_kg', 'waste_landfill_kg',
    'is_holiday', 'has_special_event',
    'year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'quarter', 'is_weekend',
    'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos'
]

full_df = daily[full_cols].copy()
full_df.to_csv('data/waste_features_full.csv', index=False)
print(f"Saved waste_features_full.csv with shape {full_df.shape}")

Saved waste_features_full.csv with shape (2618, 21)


## Lag Features for XGBoost and LightGBM models

In [8]:
daily_sorted = daily.sort_values(['section', 'date']).copy()
lags = [1, 7, 14, 28]
for lag in lags:
    daily_sorted[f'lag_{lag}'] = daily_sorted.groupby('section')['waste_kg'].shift(lag)

print("Lag features added.")
daily_sorted[['section', 'date', 'waste_kg', 'lag_1', 'lag_7', 'lag_14', 'lag_28']].head(10)

Lag features added.


,section,date,waste_kg,lag_1,lag_7,lag_14,lag_28
0,a,2025-01-01,0.1800,NaN,NaN,NaN,NaN
1,a,2025-01-01,3.1800,0.1800,NaN,NaN,NaN
2,a,2025-01-01,4.2300,3.1800,NaN,NaN,NaN
14,a,2025-01-02,0.0600,4.2300,NaN,NaN,NaN
15,a,2025-01-02,1.5000,0.0600,NaN,NaN,NaN
16,a,2025-01-02,1.8200,1.5000,NaN,NaN,NaN
29,a,2025-01-03,1.7100,1.8200,NaN,NaN,NaN
30,a,2025-01-03,0.1200,1.7100,0.1800,NaN,NaN
31,a,2025-01-03,3.3400,0.1200,3.1800,NaN,NaN
32,a,2025-01-03,2.5600,3.3400,4.2300,NaN,NaN


## Rolling Window Statistics for XGBoost and LightGBM models

In [9]:
windows = [7, 14]
for w in windows:
    daily_sorted[f'rolling_mean_{w}'] = daily_sorted.groupby('section')['waste_kg'].transform(
        lambda x: x.rolling(w, min_periods=1).mean()
    )

daily_sorted['rolling_std_7'] = daily_sorted.groupby('section')['waste_kg'].transform(
    lambda x: x.rolling(7, min_periods=1).std()
)

daily_sorted['rolling_max_7'] = daily_sorted.groupby('section')['waste_kg'].transform(
    lambda x: x.rolling(7, min_periods=1).max()
)

print("Rolling features added.")
daily_sorted[['section', 'date', 'waste_kg', 'rolling_mean_7', 'rolling_std_7', 'rolling_max_7']].head(10)

Rolling features added.


,section,date,waste_kg,rolling_mean_7,rolling_std_7,rolling_max_7
0,a,2025-01-01,0.1800,0.1800,NaN,0.1800
1,a,2025-01-01,3.1800,1.6800,2.1213,3.1800
2,a,2025-01-01,4.2300,2.5300,2.1018,4.2300
14,a,2025-01-02,0.0600,1.9125,2.1143,4.2300
15,a,2025-01-02,1.5000,1.8300,1.8403,4.2300
16,a,2025-01-02,1.8200,1.8283,1.6460,4.2300
29,a,2025-01-03,1.7100,1.8114,1.5033,4.2300
30,a,2025-01-03,0.1200,1.8029,1.5143,4.2300
31,a,2025-01-03,3.3400,1.8257,1.5395,4.2300
32,a,2025-01-03,2.5600,1.5871,1.1959,3.3400


## Location Encoding for XGBoost and LightGBM models

In [10]:
section_mapping = {'a': 0, 'b': 1, 'c': 2, 'd': 3}
daily_sorted['section_encoded'] = daily_sorted['section'].map(section_mapping)
one_hot = pd.get_dummies(daily_sorted['section'], prefix='section')
daily_sorted = pd.concat([daily_sorted, one_hot], axis=1)
print("Location encoding added.")
daily_sorted[['section', 'section_encoded', 'section_a', 'section_b', 'section_c', 'section_d']].head()

Location encoding added.


,section,section_encoded,section_a,section_b,section_c,section_d
0,a,0,True,False,False,False
1,a,0,True,False,False,False
2,a,0,True,False,False,False
14,a,0,True,False,False,False
15,a,0,True,False,False,False


## Handle NaNs from Lag and Rolling Features

In [11]:
nan_counts = daily_sorted[['lag_1', 'lag_7', 'lag_14', 'lag_28']].isna().sum()
print("NaN counts before dropping:\n", nan_counts)
xgb_df = daily_sorted.dropna(subset=['lag_1', 'lag_7', 'lag_14', 'lag_28']).copy()
print(f"XGBoost dataset shape after dropping lag NaNs: {xgb_df.shape}")

NaN counts before dropping:
 lag_1       4
lag_7      28
lag_14     56
lag_28    112
dtype: int64
XGBoost dataset shape after dropping lag NaNs: (2506, 35)


## Final XGBoost and LightGBM Dataset

In [12]:
exclude_cols = ['date', 'section']
feature_cols = [col for col in xgb_df.columns if col not in exclude_cols]

if 'waste_kg' not in feature_cols:
    feature_cols = ['waste_kg'] + feature_cols

xgb_export = xgb_df[feature_cols].copy()
xgb_export.to_csv('data/waste_features_xgb.csv', index=False)
print(f"Saved waste_features_xgb.csv with shape {xgb_export.shape}")
print("Features:", list(xgb_export.columns))

Saved waste_features_xgb.csv with shape (2506, 33)
Features: ['meal', 'waste_kg', 'waste_organic_kg', 'waste_recyclable_kg', 'waste_landfill_kg', 'foot_traffic', 'is_holiday', 'has_special_event', 'year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'quarter', 'is_weekend', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_14', 'rolling_std_7', 'rolling_max_7', 'section_encoded', 'section_a', 'section_b', 'section_c', 'section_d']
